In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

c:\Users\plunm\personal\course\ml-zoomcamp2025\.venv\Lib\site-packages\torch\cuda\__init__.py:182: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
print(torch.__version__)

2.8.0+cu126


In [3]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        
        # Convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=3,    # RGB
            out_channels=32,  # number of filters
            kernel_size=3     # 3x3 filter
        )
        
        # Max Pooling
        self.pool = nn.MaxPool2d(2, 2)  # kernel_size=2, stride=2
        
        # Fully connected layers
        # Flatten size: 32 * 99 * 99 = 313632
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.fc2 = nn.Linear(64, 1)  # output 1 neuron

    def forward(self, x):
        # Conv + ReLU
        x = F.relu(self.conv1(x))
        
        # MaxPool
        x = self.pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1) 
        
        # Fully connected + ReLU
        x = F.relu(self.fc1(x))
        
        x = self.fc2(x) 
        
        return x

In [4]:
model = CNNModel()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8
)

criterion = nn.BCEWithLogitsLoss()

print(model)

CNNModel(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=313632, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [5]:
# Option 1: Using torchsummary (install with: pip install torchsummary)
from torchsummary import summary
summary(model, input_size=(3, 200, 200)) 

# Option 2: Manual counting
""" total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}") """

Layer (type:depth-idx)                   Param #
├─Conv2d: 1-1                            896
├─MaxPool2d: 1-2                         --
├─Linear: 1-3                            20,072,512
├─Linear: 1-4                            65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0


' total_params = sum(p.numel() for p in model.parameters())\nprint(f"Total parameters: {total_params}") '

In [6]:
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) 
])

test_transforms = train_transforms

In [7]:
train_dataset = datasets.ImageFolder(root="data/train", transform=train_transforms)
validation_dataset = datasets.ImageFolder(root="data/test", transform=test_transforms)

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=20,
    shuffle=False
)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CNNModel(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=313632, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)

In [10]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

In [11]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6462, Acc: 0.6362, Val Loss: 0.6032, Val Acc: 0.6517
Epoch 2/10, Loss: 0.5475, Acc: 0.7100, Val Loss: 0.7251, Val Acc: 0.6318
Epoch 3/10, Loss: 0.5533, Acc: 0.7250, Val Loss: 0.5991, Val Acc: 0.6716
Epoch 4/10, Loss: 0.4802, Acc: 0.7712, Val Loss: 0.6033, Val Acc: 0.6567
Epoch 5/10, Loss: 0.4334, Acc: 0.8025, Val Loss: 0.6196, Val Acc: 0.6766
Epoch 6/10, Loss: 0.3740, Acc: 0.8325, Val Loss: 0.7371, Val Acc: 0.6766
Epoch 7/10, Loss: 0.2721, Acc: 0.8838, Val Loss: 0.9223, Val Acc: 0.6418
Epoch 8/10, Loss: 0.2478, Acc: 0.9000, Val Loss: 0.7294, Val Acc: 0.7214
Epoch 9/10, Loss: 0.2075, Acc: 0.9200, Val Loss: 0.7523, Val Acc: 0.7015
Epoch 10/10, Loss: 0.1494, Acc: 0.9450, Val Loss: 0.7894, Val Acc: 0.7015


In [12]:
median_acc = np.median(history['acc'])
median_acc

np.float64(0.8175)

In [13]:
std_loss = np.std(history['loss'])
std_loss

np.float64(0.15896409829846403)

# Data Augmentation

In [14]:
train_transforms = transforms.Compose([
    transforms.RandomRotation(50),                        
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)), 
    transforms.RandomHorizontalFlip(),                     
    transforms.ToTensor(),                                 
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],                      
        std=[0.229, 0.224, 0.225]
    )
])


In [15]:
train_dataset = datasets.ImageFolder(root="data/train", transform=train_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True
)

In [16]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.7418, Acc: 0.6275, Val Loss: 0.6577, Val Acc: 0.6816
Epoch 2/10, Loss: 0.5797, Acc: 0.6837, Val Loss: 0.7310, Val Acc: 0.6766
Epoch 3/10, Loss: 0.5754, Acc: 0.7063, Val Loss: 0.5767, Val Acc: 0.6965
Epoch 4/10, Loss: 0.5522, Acc: 0.7238, Val Loss: 0.5689, Val Acc: 0.7015
Epoch 5/10, Loss: 0.5593, Acc: 0.7013, Val Loss: 0.6367, Val Acc: 0.6617
Epoch 6/10, Loss: 0.5234, Acc: 0.7425, Val Loss: 0.5793, Val Acc: 0.7164
Epoch 7/10, Loss: 0.5254, Acc: 0.7350, Val Loss: 0.7738, Val Acc: 0.6070
Epoch 8/10, Loss: 0.5033, Acc: 0.7562, Val Loss: 0.8803, Val Acc: 0.6169
Epoch 9/10, Loss: 0.5048, Acc: 0.7350, Val Loss: 0.5254, Val Acc: 0.7313
Epoch 10/10, Loss: 0.4723, Acc: 0.7863, Val Loss: 0.5510, Val Acc: 0.7363


In [19]:
history

{'acc': [0.63625,
  0.71,
  0.725,
  0.77125,
  0.8025,
  0.8325,
  0.88375,
  0.9,
  0.92,
  0.945,
  0.6275,
  0.68375,
  0.70625,
  0.72375,
  0.70125,
  0.7425,
  0.735,
  0.75625,
  0.735,
  0.78625],
 'loss': [0.6462259121239186,
  0.5475101299583912,
  0.5532774798572063,
  0.48020908832550047,
  0.4333876986056566,
  0.3740024916827679,
  0.27212331630289555,
  0.24781085457652807,
  0.20747365271672608,
  0.14940991019830108,
  0.7417503699660302,
  0.5797114685177803,
  0.5754108980298043,
  0.5522442437708378,
  0.5592503190040589,
  0.5233907453715801,
  0.5253977030515671,
  0.5032677464187145,
  0.5047888875007629,
  0.47233611568808553],
 'val_acc': [0.6517412935323383,
  0.6318407960199005,
  0.6716417910447762,
  0.6567164179104478,
  0.6766169154228856,
  0.6766169154228856,
  0.6417910447761194,
  0.7213930348258707,
  0.7014925373134329,
  0.7014925373134329,
  0.681592039800995,
  0.6766169154228856,
  0.6965174129353234,
  0.7014925373134329,
  0.6616915422885572,

In [22]:
mean_test_loss = np.mean(history['val_loss'])
mean_test_loss

np.float64(0.6780760571212319)

In [23]:
avg_last5_val_acc = np.mean(history['val_acc'][15:20])
avg_last5_val_acc

np.float64(0.681592039800995)